# dk EGAT 모델 풀 학습 (Colab A100)

> **최종 업데이트**: 2026-07-14: 제안 아키텍처(BERT + ATLOP LCP + 2-Layer Edge-featured GAT +
> BCE + 0.2×Evidence Contrastive) 풀 학습 노트북. distant_epochs/annotated epochs/seed를
> `Scripts/atlop` baseline과 동일하게 맞춰 아키텍처 변수만 비교하도록 수정. 파이프라인 상세는
> `Scripts/dk_gat/README.md`.

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: distant 20,000개 × **1 epoch** 프리트레인 → annotated 3,053개 × **15 epoch**
파인튜닝 (AdamW 2e-5, wd 0.01, warmup 6%, clip 1.0, seed 66) — **`Scripts/atlop` baseline과
distant_limit/distant_epochs/epochs/seed를 전부 동일하게 맞춰서 GAT 추가라는 아키텍처 변수
하나만 비교**함 (원 스펙의 "2~3ep distant / 12~15ep annotated"는 범위 제안이었고, 통제 비교를
위해 baseline과 정확히 일치하는 1ep/15ep로 확정). epoch마다 dev threshold sweep으로 F1 최대
지점을 골라 `dev_F1 / Ign_F1 / threshold`를 출력.

**비교 기준** (`results/comparison.md`, 전부 동일 스코어러·seed 66·distant 20k·distant_epochs 1):

| 모델 | annotated epochs | dev F1 | Ign F1 |
|---|---|---|---|
| ATLOP baseline | 15 | 61.71 | 59.86 |
| ATLOP + PUATLoss(0.7) | 15 | 62.06 | 60.16 |
| **dk EGAT (이 노트북)** | 15 | | |

In [ ]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

In [ ]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git lfs pull --include="docred_data/data/dev.json,docred_data/data/train_annotated.json,docred_data/data/train_distant.json,docred_data/data/rel_info.json"
# json들이 MB~GB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*.json

In [ ]:
# 2) 패키지
!pip install -q transformers==4.57.6 accelerate

In [ ]:
# 3) 학습: distant 20k×1ep -> annotated 15ep (baseline과 동일 스케줄, A100 약 2~3시간)
#    epoch마다 [스테이지 | epoch N] dev_F1=.. Ign_F1=.. threshold=.. 로그가 찍힘
!python -m Scripts.dk_gat.train_gat \
  --distant_limit 20000 --distant_epochs 1 --epochs 15 \
  --lr 2e-5 --weight_decay 0.01 --warmup_ratio 0.06 --dropout 0.1 \
  --run_name dk_gat --save_model --seed 66

In [ ]:
# 4) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat.pt results/dk_gat_stage1.pt results/dk_gat_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat

## 결과 기록

마지막 epoch 로그 수치를 위의 비교표와 `results/comparison.md`에 반영.

- threshold sweep이 고른 값(로그의 `threshold=..`)도 함께 기록 — 최종 예측 재현에 필요.
- stage-1(distant 직후) 수치도 따로 적어두면 GAT가 노이즈 라벨 위에서 얼마나 버티는지,
  annotated 파인튜닝이 얼마나 끌어올리는지 분해해 보여줄 수 있음.
- seed 66 단일 시드 — baseline과의 차이가 ±1점 이내면 PRD §5 시드 노이즈 주의 적용.
- Evidence Contrastive Loss는 annotated 단계에서만 실질 작동함(distant는 evidence 없음).
  GAT 고도화 실험(edge feature ablation, 헤드 수, 인접행렬 반경 등)은 이 노트북의 셀 3에
  `--gat_heads`, `--evidence_weight` 인자만 바꿔 반복하면 됨.